# 10 | Campaign-ready windows: the 2026 ATP calendar scored for India

**Author: Chanakya**

Every 2026 ATP tour event in FanCode’s package, split into three viewing windows (day session, night session, final) and scored for an Indian audience. A window is **campaign-ready** only if its start sits inside every one of the three viewing windows under every start delay from 0 to 120 minutes (12 of 12 tests), the same standard applied to the verified finals in notebook 01. Each window carries its clash check, player-story trigger, offer and timing confidence. The logic lives in `models/campaign_windows.py`. Player draws are not scored: they are unknown until the week of the event.

In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'data/manifests/release.json').exists())
sys.path.insert(0, str(ROOT))
from src.analysis_common import *
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
rng = np.random.default_rng(CFG['seed'])
print('Offline inputs:', CFG['raw_release'], '| Author: Chanakya')
import src.analysis_common as shared
shared.ACTIVE_NOTEBOOK='10_campaign_ready_windows'
shared.ACTIVE_SOURCES=[]

In [ ]:
sys.path.insert(0, str(ROOT/'models'))
import campaign_windows as cw
windows, events = cw.build()
summary = cw.summarise(windows, events)
display(table(windows,'10_campaign_ready_windows'))
(ROOT/'outputs/reports/10_campaign_windows_summary.json').write_text(json.dumps({'settings':cw.SETTINGS,'summary':summary},indent=1,default=str)+'\n')
settings=pd.DataFrame([dict(setting=k,value=json.dumps(v) if not isinstance(v,str) else v) for k,v in cw.SETTINGS.items()]);display(table(settings,'10_window_scoring_settings'))

## 1. How the season divides

In [ ]:
counts=windows.groupby(['session','status']).size().unstack(fill_value=0);display(counts)
st=windows.status.value_counts().rename_axis('status').reset_index(name='windows');display(table(st,'10_window_status_counts'))
print(f"{summary['campaign_ready_live']} of {summary['windows']} windows across {summary['events']} events are campaign-ready; {summary['upcoming_live_windows']} are still ahead after 18 September 2026.")

In [ ]:
colors={'Campaign-ready: sell live':COLORS[1],'Early evening: live with start reminder':COLORS[0],'Late: remind + replay':COLORS[2],'Overnight: replay only':COLORS[3],'Daytime: highlights':COLORS[5]}
size={'Finals':140,'Masters 1000':110,'500':70,'250':40}
fig_,ax=plt.subplots(figsize=(12,4.8))
w=windows.copy();w['d']=pd.to_datetime(w.window_date);w['h']=[int(x[:2])+int(x[3:])/60 for x in w.ist_start]
for s_,g in w.groupby('status'):ax.scatter(g.d,g.h,s=[size[t] for t in g.tier],color=colors[s_],alpha=.8,label=s_,edgecolor='white',linewidth=.5)
ax.axhspan(18,23,color=COLORS[1],alpha=.08);ax.set_ylim(0,24);ax.set_yticks(range(0,25,3));ax.set_ylabel('Start, IST hour');ax.legend(fontsize=8,loc='upper center',bbox_to_anchor=(.5,-.08),ncol=5,frameon=False)
ax.axvline(pd.Timestamp('2026-09-18'),color='black',ls=':');ax.text(pd.Timestamp('2026-09-20'),1,'today',fontsize=8)
ax.set_title('The Europe and Gulf swing lands in Indian prime time; the Americas and Asia do not')
fig('10_campaign_windows_calendar','Marker size = tier. Shaded band = 18:00-23:00 IST. Session times are modelled local conventions except the 12 finals verified from official orders of play.')

## 2. The list: campaign-ready windows, ranked
Windows are ranked by status first, then by score. Score out of 100 = 40% timing robustness + 30% tier + 15% player-story continuity (post-Slam follow-through, race to Turin, season finale) + 15% clash-free. Weights are settings, not estimates. A clash does not disqualify a window: it changes who receives the send.

In [ ]:
ready=windows[windows.status=='Campaign-ready: sell live'][['rank','event','tier','session','window_date','ist_start','clash_check','continuity_trigger','offer','timing_confidence','upcoming','score']]
display(table(ready,'10_campaign_ready_list'))
ahead=ready[ready.upcoming];display(table(ahead,'10_campaign_ready_upcoming'))

## 3. Checks

In [ ]:
checks={'fifty_five_dated_events':summary['events']==55,
 'three_windows_per_event':len(windows)==3*summary['events'],
 'status_counts_reconcile':sum(v for k,v in summary.items() if k in ['campaign_ready_live','early_evening_live','late_remind_replay','daytime_highlights','overnight_replay'])==len(windows),
 'verified_finals_used':summary['high_confidence_windows']==12,
 'rotterdam_final_2000_ist':windows[(windows.event=='Rotterdam')&(windows.session=='Final')].ist_start.iloc[0]=='20:00',
 'indian_wells_final_overnight':windows[(windows.event=='Indian Wells')&(windows.session=='Final')].status.iloc[0]=='Overnight: replay only',
 'six_verified_finals_robust':int(((windows.session=='Final')&windows.timing_confidence.str.startswith('High')&(windows.status=='Campaign-ready: sell live')).sum())==6,
 'unique_ranks':windows['rank'].is_unique}
check('10_campaign_windows',checks)
top=', '.join(f"{r.event} {r.session.lower()} {r.window_date} {r.ist_start} IST" for r in ahead.head(8).itertuples())
report('10_campaign_windows_findings',f"Of {summary['windows']} windows across {summary['events']} 2026 ATP events in FanCode's package, {summary['campaign_ready_live']} are campaign-ready: their start holds inside Indian prime time under every timing test. {summary['early_evening_live']} more start in the early evening and can be sold live with a start reminder; {summary['late_remind_replay']} are late and get reminders plus the INR 39 replay; {summary['overnight_replay']} are overnight and {summary['daytime_highlights']} daytime. {summary['upcoming_live_windows']} campaign-ready windows are still ahead this season: {top}. Timing confidence is high only for the 12 finals verified from official orders of play; the rest use modelled local session times and must be confirmed from each week's order of play before a send. Football clashes after 8 September 2026 are not yet checked because 2026-27 fixtures were not captured.")